In [19]:
# Regime Analysis Notebook
# 
# Diagnóstico por janela: cruza retornos do walk-forward com regimes de mercado.

# --- Config ---
CONFIG_PATH  = "INF/config_features_32slots.yaml"
RESULTS_PATH = "INF/outputs/runs_summary.csv"
EXPERIMENT   = "feat_b4_macdh_only"  # experiment_name a analisar
OUT_DIR      = "INF/outputs/regime_analysis"

# Regime params (podes ajustar)
TREND_PERIOD = 200
ATR_FAST     = 14
ATR_SLOW     = 200
PWR_FAST     = 20
PWR_SLOW     = 200

# Thresholds (para o plot/diagnóstico textual)
TH_TREND_BULL = 0.05
TH_TREND_BEAR = -0.05
TH_VOL_HIGH   = 1.5
TH_VOL_LOW    = 0.7
TH_PWR_HIGH   = 1.5

In [20]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterator, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml


def find_project_root(start: Optional[Path] = None) -> Path:
    """Find repo root (the directory that contains INF/)."""
    p = (start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / "INF").is_dir():
            return cand
    return p


PROJECT_ROOT = find_project_root()


def resolve_path(p: str | Path) -> Path:
    pp = Path(p)
    if pp.is_absolute():
        return pp
    return (PROJECT_ROOT / pp).resolve()


print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\jtoma\projects\NN


In [21]:
# --- Regime indicators (computed on full series, no lookahead) ---

def regime_trend(closes: np.ndarray, period: int = 200) -> np.ndarray:
    """(close - MA200) / MA200 — posição relativa à tendência longa."""
    s = pd.Series(closes)
    ma = s.rolling(period, min_periods=period).mean().to_numpy()
    with np.errstate(invalid="ignore", divide="ignore"):
        out = np.where(ma > 0, (closes - ma) / ma, np.nan)
    return out


def _atr_series(highs: np.ndarray, lows: np.ndarray, closes: np.ndarray, period: int) -> np.ndarray:
    tr = np.maximum(
        highs[1:] - lows[1:],
        np.maximum(np.abs(highs[1:] - closes[:-1]), np.abs(lows[1:] - closes[:-1])),
    )
    tr = np.concatenate([[np.nan], tr])
    return pd.Series(tr).rolling(period, min_periods=period).mean().to_numpy()


def regime_vol(
    highs: np.ndarray,
    lows: np.ndarray,
    closes: np.ndarray,
    fast: int = 14,
    slow: int = 200,
) -> np.ndarray:
    """ATR(14) / ATR(200) — volatilidade relativa (ratio > 1 = pânico)."""
    atr_fast = _atr_series(highs, lows, closes, fast)
    atr_slow = _atr_series(highs, lows, closes, slow)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(atr_slow > 0, atr_fast / atr_slow, np.nan)


def regime_pwr(volumes: np.ndarray, fast: int = 20, slow: int = 200) -> np.ndarray:
    """Vol(20) / VolMédia(200) — pressão de volume relativa."""
    s = pd.Series(volumes)
    fast_ma = s.rolling(fast, min_periods=fast).mean().to_numpy()
    slow_ma = s.rolling(slow, min_periods=slow).mean().to_numpy()
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(slow_ma > 0, fast_ma / slow_ma, np.nan)

In [22]:
# --- Walk-forward window slicer (mirrors INF/data_loader.py) ---

@dataclass(frozen=True)
class WalkWindow:
    window_id: int
    train_start: int
    train_end: int
    val_start: int
    val_end: int
    test_start: Optional[int] = None
    test_end: Optional[int] = None


def iter_walkforward_windows(
    df: pd.DataFrame,
    train_size: int,
    val_size: int,
    step_size: int,
    *,
    test_size: Optional[int] = None,
    anchor: int = 0,
) -> Iterator[WalkWindow]:
    if train_size <= 0 or val_size <= 0 or step_size <= 0:
        raise ValueError("train_size, val_size e step_size devem ser > 0")
    if anchor < 0:
        raise ValueError("anchor deve ser >= 0")
    if test_size is not None and test_size < 0:
        raise ValueError("test_size deve ser null ou >= 0")

    n = len(df)
    ts = test_size if test_size is not None else 0

    k = 0
    while True:
        train_start = anchor + k * step_size
        train_end = train_start + train_size
        val_start = train_end
        val_end = val_start + val_size

        if ts > 0:
            test_start = val_end
            test_end = test_start + ts
            if train_start < 0 or test_end > n:
                break
            yield WalkWindow(
                window_id=k,
                train_start=train_start,
                train_end=train_end,
                val_start=val_start,
                val_end=val_end,
                test_start=test_start,
                test_end=test_end,
            )
        else:
            if train_start < 0 or val_end > n:
                break
            yield WalkWindow(
                window_id=k,
                train_start=train_start,
                train_end=train_end,
                val_start=val_start,
                val_end=val_end,
                test_start=None,
                test_end=None,
            )
        k += 1

In [23]:
# --- Load config + data, compute per-window table, save CSV ---

# Config
cfg_path = resolve_path(CONFIG_PATH)
with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

data_cfg = cfg["data"]
wf_cfg = cfg["walkforward"]

# Data
csv_path = resolve_path(data_cfg["csv_path"])
df = pd.read_csv(csv_path)

recent = int(data_cfg.get("recent_rows", 0) or 0)
if recent > 0:
    df = df.tail(recent).reset_index(drop=True)

for col in ("open", "high", "low", "close", "volume"):
    if col not in df.columns:
        raise ValueError(f"Missing required column '{col}' in {csv_path}")

closes = df["close"].to_numpy(dtype=float)
highs = df["high"].to_numpy(dtype=float)
lows = df["low"].to_numpy(dtype=float)
volumes = df["volume"].to_numpy(dtype=float)

# Regimes (full series)
r_trend = regime_trend(closes, period=TREND_PERIOD)
r_vol = regime_vol(highs, lows, closes, fast=ATR_FAST, slow=ATR_SLOW)
r_pwr = regime_pwr(volumes, fast=PWR_FAST, slow=PWR_SLOW)

# Results
results_path = resolve_path(RESULTS_PATH)
results_df = pd.read_csv(results_path)
exp_df = results_df[results_df["experiment_name"] == EXPERIMENT].copy()
if exp_df.empty:
    raise ValueError(f"Experiment '{EXPERIMENT}' not found in {results_path}")

# take most recent row for that experiment (created_at is ISO-like string)
row = exp_df.sort_values("created_at").iloc[-1]

# Windows
windows = list(
    iter_walkforward_windows(
        df,
        train_size=int(wf_cfg["train_size"]),
        val_size=int(wf_cfg["val_size"]),
        step_size=int(wf_cfg["step_size"]),
        test_size=int(wf_cfg.get("test_size") or 0) or None,
        anchor=int(wf_cfg.get("anchor", 0) or 0),
    )
)

records = []
for w in windows:
    col = f"window_{w.window_id:03d}_return"
    ret = float(row[col]) if col in row.index and pd.notna(row[col]) else np.nan

    if w.test_start is None or w.test_end is None:
        # fallback: treat val as 'test' if config has no test_size
        sl = slice(w.val_start, w.val_end)
        test_start, test_end = w.val_start, w.val_end
    else:
        sl = slice(w.test_start, w.test_end)
        test_start, test_end = w.test_start, w.test_end

    records.append(
        {
            "window": w.window_id,
            "test_start": int(test_start),
            "test_end": int(test_end),
            "return": ret,
            "regime_trend": float(np.nanmean(r_trend[sl])),
            "regime_vol": float(np.nanmean(r_vol[sl])),
            "regime_pwr": float(np.nanmean(r_pwr[sl])),
        }
    )

table = pd.DataFrame.from_records(records)

# Save
out_dir = resolve_path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

csv_out = out_dir / f"{EXPERIMENT}_regime_table.csv"
table.to_csv(csv_out, index=False, float_format="%.6f")

# Display (sorted by window)
display(table.sort_values("window"))
print("Saved table:", csv_out)

,window,test_start,test_end,return,regime_trend,regime_vol,regime_pwr
0,0,5700,6400,NaN,-0.011023,0.947182,0.903010
1,1,6400,7100,NaN,-0.008543,1.044114,1.066121
2,2,7100,7800,NaN,-0.044560,1.034493,1.121173
3,3,7800,8500,NaN,-0.029115,0.909754,0.909854
4,4,8500,9200,NaN,-0.005802,0.992795,1.032000
5,5,9200,9900,NaN,0.055964,1.083079,1.046469
6,6,9900,10600,NaN,-0.011935,0.998368,0.979817
7,7,10600,11300,NaN,0.063273,1.066447,1.040121
8,8,11300,12000,NaN,0.030402,1.035266,1.011671
9,9,12000,12700,NaN,-0.004496,0.926934,0.904634


Saved table: C:\Users\jtoma\projects\NN\INF\outputs\regime_analysis\feat_b4_macdh_only_regime_table.csv


In [24]:
# --- Plot (4 stacked panels) + save PNG ---

def plot_regime_table(table: pd.DataFrame, exp_name: str, out_dir: Path) -> Path:
    table = table.sort_values("window")

    wins = table["window"].to_numpy()
    rets = table["return"].to_numpy(dtype=float)

    fig = plt.figure(figsize=(16, 12))
    fig.suptitle(f"Regime Analysis — {exp_name}", fontsize=14, fontweight="bold")
    gs = gridspec.GridSpec(4, 1, hspace=0.55)

    # Return
    ax0 = fig.add_subplot(gs[0])
    colors = ["#e74c3c" if (not np.isnan(r) and r < 0) else "#2ecc71" for r in rets]
    ax0.bar(wins, rets * 100.0, color=colors, alpha=0.85)
    ax0.axhline(0, color="black", linewidth=0.8, alpha=0.6)
    ax0.set_ylabel("Return (%)")
    ax0.set_title("Retorno por janela (test split)")

    # Trend
    ax1 = fig.add_subplot(gs[1])
    ax1.bar(wins, table["regime_trend"], color="#3498db", alpha=0.75)
    ax1.axhline(TH_TREND_BULL, color="#2ecc71", linewidth=1, linestyle="--", label=f"Bullish (>{TH_TREND_BULL})")
    ax1.axhline(TH_TREND_BEAR, color="#e74c3c", linewidth=1, linestyle="--", label=f"Bearish (<{TH_TREND_BEAR})")
    ax1.set_ylabel("(close-MA)/MA")
    ax1.set_title("Regime_Trend")
    ax1.legend(fontsize=8)

    # Vol
    ax2 = fig.add_subplot(gs[2])
    ax2.bar(wins, table["regime_vol"], color="#e67e22", alpha=0.75)
    ax2.axhline(1.0, color="black", linewidth=0.8, linestyle=":", alpha=0.6)
    ax2.axhline(TH_VOL_HIGH, color="#e74c3c", linewidth=1, linestyle="--", label=f"High vol (>{TH_VOL_HIGH})")
    ax2.axhline(TH_VOL_LOW, color="#2ecc71", linewidth=1, linestyle="--", label=f"Low vol (<{TH_VOL_LOW})")
    ax2.set_ylabel("ATRfast / ATRslow")
    ax2.set_title("Regime_Vol")
    ax2.legend(fontsize=8)

    # Power
    ax3 = fig.add_subplot(gs[3])
    ax3.bar(wins, table["regime_pwr"], color="#9b59b6", alpha=0.75)
    ax3.axhline(1.0, color="black", linewidth=0.8, linestyle=":", alpha=0.6)
    ax3.axhline(TH_PWR_HIGH, color="#e74c3c", linewidth=1, linestyle="--", label=f"High power (>{TH_PWR_HIGH})")
    ax3.set_xlabel("Window ID")
    ax3.set_ylabel("Volfast / Volslow")
    ax3.set_title("Regime_Pwr")
    ax3.legend(fontsize=8)

    out_path = out_dir / f"{exp_name}_regime_plot.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return out_path


png_out = plot_regime_table(table, EXPERIMENT, out_dir)
print("Saved plot:", png_out)

# Quick text diagnostics (optional)
print("\n Win  Return   Trend    Vol    Pwr   Flags")
print("-" * 60)
for _, row in table.sort_values("window").iterrows():
    flags = []
    if np.isfinite(row["regime_vol"]) and row["regime_vol"] > TH_VOL_HIGH:
        flags.append("HIGH_VOL")
    if np.isfinite(row["regime_vol"]) and row["regime_vol"] < TH_VOL_LOW:
        flags.append("LOW_VOL")
    if np.isfinite(row["regime_trend"]) and row["regime_trend"] < TH_TREND_BEAR:
        flags.append("BEARISH")
    if np.isfinite(row["regime_trend"]) and row["regime_trend"] > TH_TREND_BULL:
        flags.append("BULLISH")
    if np.isfinite(row["regime_pwr"]) and row["regime_pwr"] > TH_PWR_HIGH:
        flags.append("HIGH_PWR")

    ret = row["return"]
    ret_str = "nan" if not np.isfinite(ret) else f"{ret:>7.2%}"
    print(
        f"{int(row['window']):>3}  {ret_str:>8}  {row['regime_trend']:>7.3f}  "
        f"{row['regime_vol']:>6.3f}  {row['regime_pwr']:>6.3f}  {', '.join(flags) or '-'}"
    )

Saved plot: C:\Users\jtoma\projects\NN\INF\outputs\regime_analysis\feat_b4_macdh_only_regime_plot.png

 Win  Return   Trend    Vol    Pwr   Flags
------------------------------------------------------------
  0       nan   -0.011   0.947   0.903  -
  1       nan   -0.009   1.044   1.066  -
  2       nan   -0.045   1.034   1.121  -
  3       nan   -0.029   0.910   0.910  -
  4       nan   -0.006   0.993   1.032  -
  5       nan    0.056   1.083   1.046  BULLISH
  6       nan   -0.012   0.998   0.980  -
  7       nan    0.063   1.066   1.040  BULLISH
  8       nan    0.030   1.035   1.012  -
  9       nan   -0.004   0.927   0.905  -
 10       nan   -0.015   1.057   1.055  -
 11       nan   -0.034   1.006   1.035  -
 12       nan   -0.005   0.963   0.956  -
 13       nan    0.015   0.938   0.931  -
 14       nan   -0.067   1.019   1.108  BEARISH
 15       nan    0.010   1.008   0.997  -
